# Rediseño en PyTorch: Stage 1 end-to-end + Stage 2 TAE + Transformer

Traducción a PyTorch del notebook de TensorFlow. La arquitectura y la lógica de entrenamiento
no cambian -- siguen siendo las mismas decisiones que ya tomamos:

1. **Stage 1 end-to-end**: la CNN (`SmallResNet18`), el encoder de texto (`TextEncoder`) y el
   embedding categórico (`CategoricalEncoder`) se ejecutan dentro del propio modelo en cada
   forward pass, así que el gradiente de CE/FITB llega hasta la CNN.
2. **TAE + Transformer**: `TypeAwareGate` (modulación FiLM condicionada por categoría, inspirada
   en Vasileva et al. 2018) antes del Transformer (Stage 2).
3. **Texto precalculado sin PCA**: se carga `glove_text_embeddings.npy` (100 dims, media de
   GloVe sin reducir) y `TextEncoder` aprende la proyección a la dimensión común del modelo.
4. **Gradient accumulation**: para entrenar con poca VRAM (8GB), igual que en la versión TF.

Cambios de fondo al pasar a PyTorch:
- `tf.data.Dataset` -> `torch.utils.data.Dataset` + `DataLoader` (la carga de imágenes ahora es
  perezosa por índice con PIL/torchvision, más natural en PyTorch que el `tf.data.map`).
- `tf.GradientTape` -> `loss.backward()` + `optimizer.step()`; gradient accumulation es incluso
  más simple (basta con no llamar a `optimizer.zero_grad()` hasta el paso N).
- `tf.train.Checkpoint` -> `torch.save`/`torch.load` de `state_dict()`.
- PyTorch tiene soporte nativo de CUDA en Windows (a diferencia de TF >= 2.11), así que si tu
  cuello de botella era "no tengo GPU en Windows", esto lo resuelve directamente.

**Nota:** revisa las rutas si tu estructura de carpetas difiere de `POLYVORE_PATH`/`IMAGES_DIR`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
import os, json, random
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"
IMAGES_DIR    = POLYVORE_PATH + "images/"
OUTPUT_DIR    = POLYVORE_PATH + "data/processed/"

TEXT_EMBED_DIM = 100   # dim de tus vectores GloVe (media de tokens, SIN PCA)
IMG_SIZE       = 224
MAX_LEN        = 8     # longitud máxima del outfit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Todas las imágenes son 224x224 y los outfits se rellenan siempre a MAX_LEN,
    # así que el tamaño de entrada a la CNN es SIEMPRE el mismo -- cudnn.benchmark
    # deja que cuDNN pruebe varios algoritmos de convolución la primera vez que ve
    # cada tamaño, y reutilice el más rápido en adelante. Sin tamaños variables,
    # esto es una ganancia gratuita (más rápido tras un pequeño warmup inicial).
    torch.backends.cudnn.benchmark = True


### 1. Embeddings de texto precalculados (media de GloVe, sin PCA)

Igual que en la versión TF: `text_preprocessing_v2.ipynb` guarda `glove_text_embeddings.npy`
(100 dims, sin PCA). Aquí solo se carga; la proyección entrenable la hace `TextEncoder` más
abajo (una `nn.Linear`, guiada por la pérdida de compatibilidad, no por PCA).

In [ ]:
TEXT_EMBEDDINGS_PATH = OUTPUT_DIR + "glove_text_embeddings.npy"
TEXT_INDEX_PATH      = OUTPUT_DIR + "glove_text_embeddings_index.json"

text_matrix = np.load(TEXT_EMBEDDINGS_PATH)          # (N, 100) -- sin PCA
with open(TEXT_INDEX_PATH) as f:
    text_index = json.load(f)                        # {pos: item_id}

assert text_matrix.shape[1] == TEXT_EMBED_DIM, (
    f"TEXT_EMBED_DIM ({TEXT_EMBED_DIM}) no coincide con la matriz cargada ({text_matrix.shape[1]}); "
    "revisa que text_preprocessing_v2 guardó los vectores sin reducir con PCA."
)

item_id2textvec = {v: text_matrix[int(k)].astype(np.float32) for k, v in text_index.items()}
print(f"Embeddings de texto cargados: {len(item_id2textvec)} ítems, dim={text_matrix.shape[1]}")


### 2. Metadatos por ítem: imagen, categoría y vector de texto

In [ ]:
df = pd.read_csv(OUTPUT_DIR + "glove_text_features.csv")
df['item_id'] = df['item_id'].astype(str)

cat_encoder = LabelEncoder()
df['cat_id'] = cat_encoder.fit_transform(df['sub_category'])
NUM_CATEGORIES = df['cat_id'].nunique()
print(f"Categorías: {NUM_CATEGORIES}")

item_id2img_path = {iid: os.path.join(IMAGES_DIR, f"{iid}.jpg") for iid in df['item_id']}
item_id2cat      = dict(zip(df['item_id'], df['cat_id']))

missing_text = [iid for iid in df['item_id'] if iid not in item_id2textvec]
print(f"Ítems sin vector de texto: {len(missing_text)}")


### 2b. Resolución de outfits oficiales (CE + FITB)

Idéntico a la versión TF -- esta parte no depende del framework, es solo parseo de ficheros.
`train.json`/`valid.json`/`test.json` traen `set_id` + `items: [{item_id, index}, ...]`, y
`compatibility_*.txt`/`fill_in_blank_*.json` referencian ítems como `"<set_id>_<index>"`.

En FITB la respuesta correcta está siempre en `answers[0]` en el fichero oficial -- se baraja
en train (recalculando `correct_idx`) y se deja en orden oficial en valid/test.

In [ ]:
def build_outfit_lookup(json_paths):
    """{'<set_id>_<index>': item_id} combinando train/valid/test.json"""
    lookup = {}
    for path in json_paths:
        with open(path) as f:
            outfits = json.load(f)
        for outfit in outfits:
            set_id = outfit["set_id"]
            for item in outfit["items"]:
                lookup[f"{set_id}_{item['index']}"] = str(item["item_id"])
    return lookup

OUTFIT_JSON_PATHS = [
    POLYVORE_PATH + "nondisjoint/train.json",
    POLYVORE_PATH + "nondisjoint/valid.json",
    POLYVORE_PATH + "nondisjoint/test.json",
]
outfit_lookup = build_outfit_lookup(OUTFIT_JSON_PATHS)
print(f"Referencias set_id_index resueltas: {len(outfit_lookup)}")


def resolve_ids(tokens):
    return [outfit_lookup[t] for t in tokens]


def load_compatibility(path):
    """'<label> <set_id>_<idx1> <set_id>_<idx2> ...' -> [{"label": int, "item_ids": [...]}]"""
    samples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            label = int(parts[0])
            item_ids = resolve_ids(parts[1:])
            samples.append({"label": label, "item_ids": item_ids})
    return samples


def load_fitb(path, shuffle_candidates=False):
    """{"question": [...], "answers": [...]} -> [{"question_ids", "answer_ids", "correct_idx"}]"""
    with open(path) as f:
        data = json.load(f)

    samples = []
    for d in data:
        question_ids = resolve_ids(d["question"])
        answer_ids   = resolve_ids(d["answers"])
        correct_idx  = 0   # answers[0] es la respuesta correcta en el formato oficial

        if shuffle_candidates:
            order = list(range(len(answer_ids)))
            rng.shuffle(order)
            answer_ids  = [answer_ids[i] for i in order]
            correct_idx = order.index(0)

        samples.append({
            "question_ids": question_ids,
            "answer_ids":   answer_ids,
            "correct_idx":  correct_idx,
        })
    return samples


ce_train = load_compatibility(POLYVORE_PATH + "nondisjoint/compatibility_train.txt")
ce_valid = load_compatibility(POLYVORE_PATH + "nondisjoint/compatibility_valid.txt")
ce_test  = load_compatibility(POLYVORE_PATH + "nondisjoint/compatibility_test.txt")

fitb_train = load_fitb(POLYVORE_PATH + "nondisjoint/fill_in_blank_train.json", shuffle_candidates=True)
fitb_valid = load_fitb(POLYVORE_PATH + "nondisjoint/fill_in_blank_valid.json", shuffle_candidates=False)
fitb_test  = load_fitb(POLYVORE_PATH + "nondisjoint/fill_in_blank_test.json",  shuffle_candidates=False)

print(f"CE   -> train: {len(ce_train)}, valid: {len(ce_valid)}, test: {len(ce_test)}")
print(f"FITB -> train: {len(fitb_train)}, valid: {len(fitb_valid)}, test: {len(fitb_test)}")


### 3. Stage 1 — Encoder multimodal end-to-end

`SmallResNet18` es la misma arquitectura que en TF (3 grupos de bloques, sin `layer4`, igual
que en Vasileva et al.), traducida a `nn.Module`. Vive dentro del modelo principal, así que
recibe gradiente en cada paso de entrenamiento.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = out + residual
        return self.relu(out)


def make_layer(in_planes, planes, num_blocks, stride=1):
    downsample = None
    if stride != 1 or in_planes != planes * BasicBlock.expansion:
        downsample = nn.Sequential(
            nn.Conv2d(in_planes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(planes * BasicBlock.expansion),
        )
    layers = [BasicBlock(in_planes, planes, stride=stride, downsample=downsample)]
    in_planes = planes * BasicBlock.expansion
    for _ in range(1, num_blocks):
        layers.append(BasicBlock(in_planes, planes))
    return nn.Sequential(*layers), in_planes


class SmallResNet18(nn.Module):
    """Igual que antes: 3 grupos de bloques (sin layer4), embedding final `embedding_size`."""
    def __init__(self, embedding_size=64):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        in_planes = 64
        self.layer1, in_planes = make_layer(in_planes, 64,  num_blocks=2, stride=1)
        self.layer2, in_planes = make_layer(in_planes, 128, num_blocks=2, stride=2)
        self.layer3, in_planes = make_layer(in_planes, 256, num_blocks=2, stride=2)

        self.avgpool  = nn.AvgPool2d(kernel_size=7)
        self.flatten  = nn.Flatten()
        # Con input 224x224: tras conv1+maxpool+layer1-3 quedan mapas de 14x14x256;
        # avgpool(7) sobre 14x14 da 2x2 -> flatten = 256*2*2 = 1024
        self.fc_embed = nn.Linear(256 * 2 * 2, embedding_size)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        return self.fc_embed(x)


class TextEncoder(nn.Module):
    """
    Recibe el vector de texto YA CALCULADO (media de GloVe, 100 dims, sin PCA,
    generado offline en text_preprocessing_v2) y aprende una proyección entrenable
    a `embedding_size`. Esta capa es la que en TAE hace de red de embedding de texto
    g(t; phi): la parte fija (GloVe + mean-pooling) queda offline porque no lleva
    gradiente, pero la proyección SÍ es entrenable y se ajusta con la pérdida de CE/FITB.
    """
    def __init__(self, embedding_size=64, input_dim=TEXT_EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, embedding_size)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, text_vec):
        return self.relu(self.proj(text_vec))


class CategoricalEncoder(nn.Module):
    """Embedding lookup entrenable (en vez de la media fija de embeddings de imagen)."""
    def __init__(self, num_categories, embedding_size=64):
        super().__init__()
        self.embedding = nn.Embedding(num_categories, embedding_size)

    def forward(self, cat_ids):
        return self.embedding(cat_ids)


class ItemEncoder(nn.Module):
    """
    Stage 1 completo: combina imagen + texto + categoría en un único 'espacio común de
    representación', proyectando cada modalidad a `common_dim` y sumando. La suma obliga
    a las tres modalidades a vivir en el mismo espacio vectorial.
    """
    def __init__(self, num_categories, common_dim=128):
        super().__init__()
        self.image_encoder = SmallResNet18(embedding_size=common_dim)
        self.text_encoder  = TextEncoder(embedding_size=common_dim)
        self.cat_encoder   = CategoricalEncoder(num_categories, embedding_size=common_dim)
        self.norm = nn.LayerNorm(common_dim, eps=1e-6)

    def forward(self, images, text_vecs, cat_ids):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(text_vecs)
        cat_emb = self.cat_encoder(cat_ids)
        item_emb = img_emb + txt_emb + cat_emb
        return self.norm(item_emb)


### 4. TAE — Type-Aware Gate (adaptado de Vasileva et al. 2018)

Igual que en la versión TF: en vez de una máscara por *par* de tipos (como en TAE original,
difícil de encajar directamente en atención sobre secuencias), cada prenda recibe una
modulación FiLM (`gamma`, `beta`) condicionada por su propia categoría antes de entrar al
Transformer.

In [ ]:
class TypeAwareGate(nn.Module):
    def __init__(self, num_categories, d_model):
        super().__init__()
        self.type_embedding = nn.Embedding(num_categories, d_model)
        self.gamma_proj = nn.Linear(d_model, d_model)
        self.beta_proj  = nn.Linear(d_model, d_model)

    def forward(self, x, cat_ids):
        t = self.type_embedding(cat_ids)
        gamma = torch.sigmoid(self.gamma_proj(t))   # gate en [0,1], estilo TAE
        beta  = self.beta_proj(t)
        return x * gamma + beta


### 5. Stage 2 — Transformer sobre embeddings type-aware

`nn.MultiheadAttention` con `batch_first=True` para no tener que transponer a `(seq, batch, dim)`
manualmente. El resto (pre/post-norm, feed-forward, dropout) es la misma estructura que en TF.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # key_padding_mask: (batch, seq) bool, True = posición a IGNORAR (igual semántica
        # que nn.MultiheadAttention espera -- coincide con nuestra "mask" de padding)
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out))


class TAETransformerOutfitModel(nn.Module):
    """
    Stage 1 (ItemEncoder end-to-end) -> TypeAwareGate (TAE) -> Transformer (Stage 2).
    La imagen se procesa cruda (CNN entrenable); el texto llega como vector de 100 dims
    (fijo, offline) y TextEncoder aprende su proyección -- el gradiente de CE/FITB llega
    hasta la CNN, la proyección de texto y los embeddings categóricos.
    """
    def __init__(self, num_categories, d_model=128, n_heads=4, ffn_dim=256,
                 n_layers=2, dropout=0.1, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.item_encoder = ItemEncoder(num_categories, common_dim=d_model)
        self.type_gate     = TypeAwareGate(num_categories, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        self.ce_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
        self.fitb_head = nn.Sequential(
            nn.Linear(d_model * 2, 128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def encode_items(self, images, text_vecs, cat_ids):
        """images: (B, L, 3, 224, 224) | text_vecs: (B, L, TEXT_EMBED_DIM) | cat_ids: (B, L)"""
        B, L = images.shape[0], images.shape[1]
        images_flat = images.reshape(B * L, 3, IMG_SIZE, IMG_SIZE)
        text_flat   = text_vecs.reshape(B * L, TEXT_EMBED_DIM)
        cats_flat   = cat_ids.reshape(B * L)

        item_emb = self.item_encoder(images_flat, text_flat, cats_flat)   # (B*L, d_model)
        item_emb = self.type_gate(item_emb, cats_flat)                     # TAE gate
        item_emb = item_emb.reshape(B, L, -1)
        return item_emb

    def encode_outfit(self, images, text_vecs, cat_ids, padding_mask=None):
        """padding_mask: (B, L) bool, True donde hay padding (a ignorar)."""
        x = self.encode_items(images, text_vecs, cat_ids)
        x = self.dropout(x)

        batch_size = x.shape[0]
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)

        if padding_mask is not None:
            cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=x.device)
            key_padding_mask = torch.cat([cls_mask, padding_mask], dim=1)
        else:
            key_padding_mask = None

        for layer in self.encoder_layers:
            x = layer(x, key_padding_mask=key_padding_mask)

        cls_out = F.normalize(x[:, 0, :], p=2, dim=-1)
        return cls_out

    def call_ce(self, images, text_vecs, cat_ids, padding_mask=None):
        cls = self.encode_outfit(images, text_vecs, cat_ids, padding_mask)
        return self.ce_head(cls)

    def call_fitb(self, q_images, q_text, q_cats, cand_images, cand_text, cand_cats, q_mask=None):
        q_cls = self.encode_outfit(q_images, q_text, q_cats, q_mask)   # (B, d_model)

        # Vectorizado: las K candidatas se procesan en UNA sola pasada de tamaño B*K,
        # en vez de un bucle Python con K llamadas de tamaño B. Resultado matemáticamente
        # idéntico (cada candidata sigue pasando por la misma CNN/TAE gate/fitb_head con
        # los mismos pesos, de forma independiente de las demás) -- solo cambia que ahora
        # es un batch más grande de una vez en lugar de K batches pequeños secuenciales,
        # que es donde se iba buena parte del tiempo de GPU (overhead de lanzar K kernels
        # pequeños por separado, en vez de uno más grande).
        B, K = cand_images.shape[0], cand_images.shape[1]
        cand_images_flat = cand_images.reshape(B * K, cand_images.shape[2], IMG_SIZE, IMG_SIZE)
        cand_text_flat   = cand_text.reshape(B * K, TEXT_EMBED_DIM)
        cand_cats_flat   = cand_cats.reshape(B * K)

        cand_emb = self.item_encoder(cand_images_flat, cand_text_flat, cand_cats_flat)  # (B*K, d_model)
        cand_emb = self.type_gate(cand_emb, cand_cats_flat)
        cand_emb = F.normalize(cand_emb, p=2, dim=-1)
        cand_emb = cand_emb.reshape(B, K, -1)                                            # (B, K, d_model)

        q_cls_expanded = q_cls.unsqueeze(1).expand(-1, K, -1)                             # (B, K, d_model)
        combined = torch.cat([q_cls_expanded, cand_emb], dim=-1)                          # (B, K, 2*d_model)
        combined_flat = combined.reshape(B * K, -1)

        scores = self.fitb_head(combined_flat)                                            # (B*K, 1)
        return scores.reshape(B, K)


### 6. Pipeline de datos — `Dataset` + `DataLoader`

En PyTorch la carga de imágenes es perezosa de forma natural: `__getitem__` se ejecuta por
índice (potencialmente en varios workers en paralelo, vía `num_workers` del `DataLoader`), a
diferencia de `tf.data.map` donde había que construir el grafo de decodificación a mano.

Cada ítem de un outfit se representa con: imagen (tensor `(3,224,224)`, normalizada con las
medias/std de ImageNet), vector de texto (100 dims, fijo) y `cat_id`. Se hace padding a
`MAX_LEN` con ceros/máscara, igual que en la versión TF.

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                                   # (3,H,W), [0,1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_image(path):
    img = Image.open(path).convert("RGB")
    return image_transform(img)


def get_item_raw(item_id):
    """Devuelve (path_imagen, vector_texto, cat_id) crudos para un item_id."""
    item_id = str(item_id)
    return item_id2img_path[item_id], item_id2textvec[item_id], item_id2cat[item_id]


def build_outfit_tensors(item_ids, max_len=MAX_LEN):
    """
    Devuelve (images, text_vecs, cat_ids, mask) con padding a max_len.
    - images    : (max_len, 3, 224, 224)
    - text_vecs : (max_len, TEXT_EMBED_DIM)
    - cat_ids   : (max_len,)
    - mask      : (max_len,) bool, True donde hay padding
    """
    n = min(len(item_ids), max_len)
    images, texts, cats = [], [], []
    for iid in item_ids[:n]:
        path, text_vec, cat_id = get_item_raw(iid)
        images.append(load_image(path))
        texts.append(torch.from_numpy(text_vec))
        cats.append(cat_id)

    pad_needed = max_len - n
    images += [torch.zeros(3, IMG_SIZE, IMG_SIZE)] * pad_needed
    texts  += [torch.zeros(TEXT_EMBED_DIM)] * pad_needed
    cats   += [0] * pad_needed
    mask    = [False] * n + [True] * pad_needed

    images_t = torch.stack(images)                 # (max_len, 3, 224, 224)
    texts_t  = torch.stack(texts).float()           # (max_len, TEXT_EMBED_DIM)
    cats_t   = torch.tensor(cats, dtype=torch.long)
    mask_t   = torch.tensor(mask, dtype=torch.bool)
    return images_t, texts_t, cats_t, mask_t


class CompatibilityDataset(Dataset):
    """samples: lista de {"label": int, "item_ids": [...]} (de load_compatibility)"""
    def __init__(self, samples, max_len=MAX_LEN):
        self.samples = samples
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        images, texts, cats, mask = build_outfit_tensors(s["item_ids"], self.max_len)
        label = torch.tensor(s["label"], dtype=torch.float32)
        return images, texts, cats, mask, label


class FITBDataset(Dataset):
    """samples: lista de {"question_ids", "answer_ids", "correct_idx"} (de load_fitb)"""
    def __init__(self, samples, max_len=MAX_LEN):
        self.samples = samples
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        q_images, q_texts, q_cats, q_mask = build_outfit_tensors(s["question_ids"], self.max_len)

        cand_images, cand_texts, cand_cats = [], [], []
        for iid in s["answer_ids"]:
            path, text_vec, cat_id = get_item_raw(iid)
            cand_images.append(load_image(path))
            cand_texts.append(torch.from_numpy(text_vec))
            cand_cats.append(cat_id)

        cand_images_t = torch.stack(cand_images)            # (4, 3, 224, 224)
        cand_texts_t  = torch.stack(cand_texts).float()      # (4, TEXT_EMBED_DIM)
        cand_cats_t   = torch.tensor(cand_cats, dtype=torch.long)
        correct       = torch.tensor(s["correct_idx"], dtype=torch.long)

        return (q_images, q_texts, q_cats, q_mask,
                cand_images_t, cand_texts_t, cand_cats_t, correct)


def make_ce_dataset_e2e(samples, batch_size=32, shuffle=True, num_workers=0):
    ds = CompatibilityDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=num_workers, pin_memory=(device.type == "cuda"))


def make_fitb_dataset_e2e(samples, batch_size=32, shuffle=True, num_workers=0):
    ds = FITBDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=num_workers, pin_memory=(device.type == "cuda"))


**Nota sobre Windows -- IMPORTANTE:** el valor por defecto de `num_workers` se dejó en `0`
precisamente por esto. En Windows, `multiprocessing` usa el método `spawn`, y cuando el código
que crea los workers vive dentro de un notebook (no en un script `.py` con
`if __name__ == "__main__":`), los workers pueden quedarse colgados **sin dar ningún error
visible** -- el proceso simplemente no avanza, ni una sola iteración, indefinidamente. Si en
algún momento subes `num_workers` para acelerar la carga y el entrenamiento deja de avanzar sin
motivo aparente, vuelve a bajarlo a `0` como primera comprobación.

### 7. Instanciación del modelo y smoke test

Mismos hiperparámetros que en la versión TF (ver conversación anterior sobre qué valores usar
y de dónde salen, dado que el paper no los especifica).

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Hiperparámetros del Transformer (Stage 2) -- escalados desde Vaswani et al. 2017
# ──────────────────────────────────────────────────────────────────
D_MODEL  = 128
N_HEADS  = 4
FFN_DIM  = 256
N_LAYERS = 2
DROPOUT  = 0.1

model = TAETransformerOutfitModel(
    num_categories=NUM_CATEGORIES,
    d_model=D_MODEL, n_heads=N_HEADS, ffn_dim=FFN_DIM, n_layers=N_LAYERS,
    dropout=DROPOUT, max_len=MAX_LEN,
).to(device)

# if device.type == "cuda":
#     model.call_ce   = torch.compile(model.call_ce)
#     model.call_fitb = torch.compile(model.call_fitb)

# --- smoke test con datos dummy, solo para verificar shapes ---
B = 2
dummy_images = torch.zeros(B, MAX_LEN, 3, IMG_SIZE, IMG_SIZE, device=device)
dummy_text   = torch.zeros(B, MAX_LEN, TEXT_EMBED_DIM, device=device)
dummy_cats   = torch.zeros(B, MAX_LEN, dtype=torch.long, device=device)
dummy_mask   = torch.tensor([[False]*6 + [True]*2, [False]*4 + [True]*4], device=device)

model.eval()
with torch.no_grad():
    ce_out = model.call_ce(dummy_images, dummy_text, dummy_cats, dummy_mask)
    print("CE output:", ce_out.shape)   # (2, 1)

    cand_images = torch.zeros(B, 4, 3, IMG_SIZE, IMG_SIZE, device=device)
    cand_text   = torch.zeros(B, 4, TEXT_EMBED_DIM, device=device)
    cand_cats   = torch.zeros(B, 4, dtype=torch.long, device=device)

    fitb_out = model.call_fitb(dummy_images, dummy_text, dummy_cats,
                                cand_images, cand_text, cand_cats, dummy_mask)
    print("FITB output:", fitb_out.shape)  # (2, 4)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales: {n_params:,} (entrenables: {n_trainable:,})")


### Hiperparámetros de entrenamiento (optimizador, batch size)

Fuente: Vasileva et al. 2018 (TAE), igual que en la versión TF -- ver conversación anterior.

In [ ]:
EMBEDDING_DIM = 64      # dim de imagen/texto/categoría antes de proyectar a D_MODEL
LEARNING_RATE = 5e-5    # Adam, igual que en el código de referencia de TAE
WEIGHT_DECAY  = 1e-4
EPOCHS        = 50      # límite superior -- el early stopping normalmente parará antes
PATIENCE      = 5       # nº de épocas sin mejorar (ni CE-AUC ni FITB-Acc) antes de parar

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


### `train_step` / `evaluate` — con gradient accumulation + mixed precision (8GB VRAM)

Misma filosofía que antes (BCE + CrossEntropy sumadas, un backward por micro-batch, gradiente
acumulado sin `zero_grad()` intermedio). Se añade **mixed precision** (`torch.autocast` +
`GradScaler`): el forward de la CNN y el Transformer se calculan en `float16` en vez de
`float32`, lo cual en una GPU con Tensor Cores (cualquier RTX) suele dar una aceleración
notable y reduce la VRAM usada -- con margen para subir `BATCH_SIZE` si te sobra memoria
después de activarlo.

La pérdida (`BCELoss`/`CrossEntropyLoss`) se calcula **fuera** del bloque `autocast`, en
`float32`: `nn.BCELoss` sobre una salida de `Sigmoid` es numéricamente inestable en `float16`
(division por valores muy cercanos a 0), así que se saca explícitamente de la región de
precisión reducida.

In [ ]:
from sklearn.metrics import roc_auc_score

BATCH_SIZE   = 8    # micro-batch real que cabe en 8GB -- ajusta según lo que veas en nvidia-smi
ACCUM_STEPS  = 4    # batch efectivo = BATCH_SIZE * ACCUM_STEPS = 32

USE_AMP = (device.type == "cuda")   # mixed precision solo tiene sentido en GPU
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

ce_loss_fn   = nn.BCELoss()                # model.ce_head ya aplica Sigmoid
fitb_loss_fn = nn.CrossEntropyLoss()       # espera logits, no aplica softmax dentro del modelo


def train_step(ce_batch, fitb_batch):
    """Un micro-batch: forward (fp16) + backward, gradiente se ACUMULA (no se hace zero_grad aquí)."""
    ce_images, ce_text, ce_cats, ce_mask, ce_labels = [t.to(device) for t in ce_batch]
    (q_images, q_text, q_cats, q_mask,
     cand_images, cand_text, cand_cats, fitb_correct) = [t.to(device) for t in fitb_batch]

    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        ce_preds    = model.call_ce(ce_images, ce_text, ce_cats, ce_mask).squeeze(-1)
        fitb_scores = model.call_fitb(q_images, q_text, q_cats, cand_images, cand_text, cand_cats, q_mask)

    # pérdida en float32, fuera de autocast (BCELoss+Sigmoid es inestable en fp16)
    loss_ce   = ce_loss_fn(ce_preds.float(), ce_labels)
    loss_fitb = fitb_loss_fn(fitb_scores.float(), fitb_correct)

    total_loss = loss_ce + loss_fitb
    # Se escala por ACCUM_STEPS ANTES del backward, y por scaler (GradScaler) para evitar
    # underflow de gradientes pequeños en fp16 -- ambos escalados son independientes y se
    # deshacen automáticamente al hacer scaler.step(optimizer).
    scaler.scale(total_loss / ACCUM_STEPS).backward()

    return loss_ce.item(), loss_fitb.item(), total_loss.item()


@torch.no_grad()
def evaluate(ce_loader, fitb_loader, verbose=False, desc="eval"):
    model.eval()

    # CE: AUC
    all_preds, all_labels = [], []
    for ce_images, ce_text, ce_cats, ce_mask, ce_labels in tqdm(ce_loader, desc=f"{desc} (CE)", leave=False):
        ce_images, ce_text = ce_images.to(device), ce_text.to(device)
        ce_cats, ce_mask   = ce_cats.to(device), ce_mask.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            preds = model.call_ce(ce_images, ce_text, ce_cats, ce_mask).squeeze(-1)
        all_preds.append(preds.float().cpu().numpy())
        all_labels.append(ce_labels.numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    ce_auc = roc_auc_score(all_labels, all_preds)

    # FITB: Accuracy
    correct_total, total = 0, 0
    all_scores, all_correct = [], []
    for (q_images, q_text, q_cats, q_mask,
         cand_images, cand_text, cand_cats, correct) in tqdm(fitb_loader, desc=f"{desc} (FITB)", leave=False):
        q_images, q_text = q_images.to(device), q_text.to(device)
        q_cats, q_mask   = q_cats.to(device), q_mask.to(device)
        cand_images, cand_text = cand_images.to(device), cand_text.to(device)
        cand_cats = cand_cats.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            scores = model.call_fitb(q_images, q_text, q_cats, cand_images, cand_text, cand_cats, q_mask)
        predicted = torch.argmax(scores, dim=1).cpu()
        correct_total += (predicted == correct).sum().item()
        total += len(correct)
        all_scores.append(scores.float().cpu().numpy())
        all_correct.append(correct.numpy())

    fitb_acc = correct_total / total

    if verbose:
        all_scores_arr  = np.concatenate(all_scores)
        all_correct_arr = np.concatenate(all_correct)
        all_predicted   = np.argmax(all_scores_arr, axis=1)
        print(f"\n  FITB predicciones:")
        print(f"    Distribución predicted : {np.bincount(all_predicted)}")
        print(f"    Distribución correct   : {np.bincount(all_correct_arr)}")
        for i in range(all_scores_arr.shape[1]):
            print(f"    Score medio posición {i} : {all_scores_arr[:, i].mean():.4f}")

    model.train()
    return ce_auc, fitb_acc


### Diagnóstico: ¿el cuello de botella es la GPU o la carga de datos?

Se descartan los primeros pasos (`N_WARMUP`) antes de medir: la primera vez que cuDNN ve un
tamaño de entrada concreto prueba varios algoritmos internos para elegir el más rápido (más
aún con `cudnn.benchmark=True`), y ese coste de "calentar" puede dominar por completo una
medición corta si no se excluye.

In [ ]:
import time

N_WARMUP        = 5
N_PROFILE_STEPS = 20

ce_probe_ds   = make_ce_dataset_e2e(ce_train[:(N_WARMUP + N_PROFILE_STEPS) * BATCH_SIZE],
                                     batch_size=BATCH_SIZE, shuffle=False)
fitb_probe_ds = make_fitb_dataset_e2e(fitb_train[:(N_WARMUP + N_PROFILE_STEPS) * BATCH_SIZE],
                                       batch_size=BATCH_SIZE, shuffle=False)

data_time, compute_time = 0.0, 0.0
t_prev = time.perf_counter()

for step, (ce_batch, fitb_batch) in enumerate(zip(ce_probe_ds, fitb_probe_ds)):
    t_data_end = time.perf_counter()

    train_step(ce_batch, fitb_batch)
    if device.type == "cuda":
        torch.cuda.synchronize()   # sin esto, el tiempo de GPU no se mide bien (las llamadas CUDA son async)
    t_now = time.perf_counter()

    if step >= N_WARMUP:   # se descartan los N_WARMUP primeros pasos (coste de calentar cuDNN)
        data_time    += t_data_end - t_prev
        compute_time += t_now - t_data_end

    t_prev = t_now

    if step + 1 >= N_WARMUP + N_PROFILE_STEPS:
        break

total = data_time + compute_time
print(f"Tiempo carga de datos (CPU) : {data_time:6.2f}s  ({data_time/total*100:5.1f}%)")
print(f"Tiempo cómputo (GPU)        : {compute_time:6.2f}s  ({compute_time/total*100:5.1f}%)")
print(f"\nTiempo medio por micro-batch (tras warmup): {total/N_PROFILE_STEPS*1000:.1f} ms")

n_batches_epoch = min(len(ce_train_ds) if 'ce_train_ds' in dir() else len(ce_train)//BATCH_SIZE,
                      len(fitb_train_ds) if 'fitb_train_ds' in dir() else len(fitb_train)//BATCH_SIZE)
est_seconds = (total / N_PROFILE_STEPS) * n_batches_epoch
print(f"Estimación época completa   : {est_seconds/60:.1f} min ({n_batches_epoch} micro-batches)")


### 8. Checkpointing — dónde se guardan los pesos

En PyTorch no hay `CheckpointManager`: se guarda directamente con `torch.save` un diccionario
con el `state_dict()` del modelo y del optimizador (para poder reanudar sin perder los momentos
de Adam), en un único fichero `.pt` por métrica.

```
POLYVORE_PATH/checkpoints/
├── best_ce.pt
└── best_fitb.pt
```

Se guardan **dos** checkpoints separados (uno por métrica), igual que en la versión TF, porque
el mejor modelo para CE-AUC no tiene por qué coincidir con el mejor para FITB-Accuracy.

In [ ]:
CKPT_DIR = POLYVORE_PATH + "checkpoints_RE/"
os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_PATH_CE   = CKPT_DIR + "best_ce.pt"
CKPT_PATH_FITB = CKPT_DIR + "best_fitb.pt"


def save_checkpoint(path, epoch, best_metric):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
    }, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt["epoch"], ckpt["best_metric"]


print(f"Checkpoints CE   -> {CKPT_PATH_CE}")
print(f"Checkpoints FITB -> {CKPT_PATH_FITB}")

# Para restaurar el mejor modelo después de entrenar:
#     epoch, best_fitb_acc = load_checkpoint(CKPT_PATH_FITB)


### 9. Datasets finales y bucle de entrenamiento (con gradient accumulation + early stopping)

`BATCH_SIZE` es el micro-batch (definido en la celda de `train_step`). Cada `ACCUM_STEPS`
micro-batches se llama a `optimizer.step()` + `optimizer.zero_grad()` -- si el número de
micro-batches de una época no es múltiplo exacto de `ACCUM_STEPS`, al final de la época se
aplica igualmente lo que quede acumulado.

**Early stopping (patience=5):** se lleva un contador de épocas sin mejora. Se resetea a 0 en
cuanto **cualquiera** de las dos métricas de validación (CE-AUC o FITB-Acc) mejora respecto a
su mejor valor visto hasta el momento; si pasan `PATIENCE` épocas seguidas sin que **ninguna**
de las dos mejore, se corta el entrenamiento. Los checkpoints de cada métrica se siguen
guardando exactamente igual que antes, independientemente de este contador.

In [ ]:
ce_train_ds = make_ce_dataset_e2e(ce_train, batch_size=BATCH_SIZE, shuffle=True)
ce_valid_ds = make_ce_dataset_e2e(ce_valid, batch_size=BATCH_SIZE, shuffle=False)

fitb_train_ds = make_fitb_dataset_e2e(fitb_train, batch_size=BATCH_SIZE, shuffle=True)
fitb_valid_ds = make_fitb_dataset_e2e(fitb_valid, batch_size=BATCH_SIZE, shuffle=False)

# Para la métrica "train" que se imprime cada época, NO se evalúa sobre las 106k/53k
# muestras completas de train (sería tan caro como otra época entera) -- se usa una
# submuestra aleatoria del mismo tamaño que valid, solo para ver la tendencia. La métrica
# que de verdad decide el early stopping y qué checkpoint guardar sigue siendo la de
# valid completo, sin cambios.
ce_train_subset   = random.sample(ce_train, min(len(ce_valid), len(ce_train)))
fitb_train_subset = random.sample(fitb_train, min(len(fitb_valid), len(fitb_train)))
ce_train_probe_ds   = make_ce_dataset_e2e(ce_train_subset, batch_size=BATCH_SIZE, shuffle=False)
fitb_train_probe_ds = make_fitb_dataset_e2e(fitb_train_subset, batch_size=BATCH_SIZE, shuffle=False)

history = {
    "epoch": [], "train_loss_ce": [], "train_loss_fitb": [],
    "train_ce_auc": [], "train_fitb_acc": [], "val_ce_auc": [], "val_fitb_acc": [],
}

best_fitb_acc, best_ce_auc = 0.0, 0.0
epochs_without_improvement = 0

print("\n=== Entrenamiento (batch efectivo = "
      f"{BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}) ===")
print(f"Early stopping: patience={PATIENCE} épocas sin mejora en CE-AUC ni FITB-Acc\n")

model.train()
for epoch in range(1, EPOCHS + 1):
    ce_loss_epoch, fitb_loss_epoch = [], []
    step_in_accum = 0
    optimizer.zero_grad()

    n_batches = min(len(ce_train_ds), len(fitb_train_ds))
    pbar = tqdm(zip(ce_train_ds, fitb_train_ds), total=n_batches,
                desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

    for ce_batch, fitb_batch in pbar:
        lce, lfitb, ltotal = train_step(ce_batch, fitb_batch)
        ce_loss_epoch.append(lce)
        fitb_loss_epoch.append(lfitb)

        # medias sobre lo acumulado hasta ahora en la época, para ver la tendencia en vivo
        pbar.set_postfix({
            "loss_ce": f"{np.mean(ce_loss_epoch):.4f}",
            "loss_fitb": f"{np.mean(fitb_loss_epoch):.4f}",
        })

        step_in_accum += 1
        if step_in_accum == ACCUM_STEPS:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_in_accum = 0

    if step_in_accum > 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    train_ce_auc, train_fitb_acc = evaluate(ce_train_probe_ds, fitb_train_probe_ds, desc="train (submuestra)")
    val_ce_auc, val_fitb_acc     = evaluate(ce_valid_ds, fitb_valid_ds, desc="valid")

    history["epoch"].append(epoch)
    history["train_loss_ce"].append(np.mean(ce_loss_epoch))
    history["train_loss_fitb"].append(np.mean(fitb_loss_epoch))
    history["train_ce_auc"].append(train_ce_auc)
    history["train_fitb_acc"].append(train_fitb_acc)
    history["val_ce_auc"].append(val_ce_auc)
    history["val_fitb_acc"].append(val_fitb_acc)

    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Loss CE: {np.mean(ce_loss_epoch):.4f} | Loss FITB: {np.mean(fitb_loss_epoch):.4f} | "
        f"Train CE AUC: {train_ce_auc:.4f} | Val CE AUC: {val_ce_auc:.4f} | "
        f"Train FITB: {train_fitb_acc:.4f} | Val FITB: {val_fitb_acc:.4f}"
    )

    improved = False

    if val_fitb_acc > best_fitb_acc:
        best_fitb_acc = val_fitb_acc
        save_checkpoint(CKPT_PATH_FITB, epoch, best_fitb_acc)
        print(f"  -> Nuevo mejor FITB: {best_fitb_acc:.4f}")
        improved = True

    if val_ce_auc > best_ce_auc:
        best_ce_auc = val_ce_auc
        save_checkpoint(CKPT_PATH_CE, epoch, best_ce_auc)
        print(f"  -> Nuevo mejor CE AUC: {best_ce_auc:.4f}")
        improved = True

    if improved:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        print(f"  (sin mejora en ninguna métrica: {epochs_without_improvement}/{PATIENCE})")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping en la época {epoch}: "
              f"{PATIENCE} épocas sin mejorar ni CE-AUC ni FITB-Acc.")
        break

print(f"\nMejor Val FITB Acc : {best_fitb_acc:.4f}")
print(f"Mejor Val CE AUC   : {best_ce_auc:.4f}")


### 10. Evaluación final en test
 
Carga los checkpoints con mejor métrica de validación (`best_ce.pt` para CE-AUC,
`best_fitb.pt` para FITB-Accuracy) y evalúa cada uno sobre el conjunto de test
correspondiente, siguiendo el mismo criterio de muestreo de negativos que en
validación (ya materializado en los ficheros oficiales `compatibility_test.txt`
y `fill_in_blank_test.json`).
"""

In [ ]:
# --- Celda de código: construir datasets de test ---
ce_test_path   = POLYVORE_PATH + "nondisjoint/compatibility_test.txt"
fitb_test_path = POLYVORE_PATH + "nondisjoint/fill_in_blank_test.json"
 
ce_test   = load_compatibility(ce_test_path)
fitb_test = load_fitb(fitb_test_path, shuffle_candidates=False)  # orden oficial, igual que en valid/test
 
print(f"CE test   : {len(ce_test)} muestras")
print(f"FITB test : {len(fitb_test)} preguntas")
 
ce_test_ds   = make_ce_dataset_e2e(ce_test, batch_size=BATCH_SIZE, shuffle=False)
fitb_test_ds = make_fitb_dataset_e2e(fitb_test, batch_size=BATCH_SIZE, shuffle=False)
 
 
# --- Celda de código: evaluar checkpoint de CE-AUC ---
print("=== Evaluando checkpoint best_ce.pt sobre test ===")
_, best_ce_metric = load_checkpoint(CKPT_PATH_CE)
print(f"Checkpoint cargado (mejor CE-AUC en validación: {best_ce_metric:.4f})")
 
ce_auc_test, fitb_acc_ce_ckpt = evaluate(ce_test_ds, fitb_test_ds, desc="test (ckpt CE)")
print(f"\nCE-AUC (test)         : {ce_auc_test:.4f}")
print(f"FITB-Accuracy (test)  : {fitb_acc_ce_ckpt:.4f}  (con checkpoint optimizado para CE, referencia)")
 
 
# --- Celda de código: evaluar checkpoint de FITB-Accuracy ---
print("=== Evaluando checkpoint best_fitb.pt sobre test ===")
_, best_fitb_metric = load_checkpoint(CKPT_PATH_FITB)
print(f"Checkpoint cargado (mejor FITB-Accuracy en validación: {best_fitb_metric:.4f})")
 
ce_auc_fitb_ckpt, fitb_acc_test = evaluate(ce_test_ds, fitb_test_ds, desc="test (ckpt FITB)")
print(f"\nCE-AUC (test)         : {ce_auc_fitb_ckpt:.4f}  (con checkpoint optimizado para FITB, referencia)")
print(f"FITB-Accuracy (test)  : {fitb_acc_test:.4f}")
 
 
# --- Celda de código: resumen final ---
print("\n" + "=" * 55)
print("RESULTADOS FINALES (TEST) — Recommendation Engine")
print("=" * 55)
print(f"{'Métrica':<30} {'Valor':>10}")
print("-" * 55)
print(f"{'CE-AUC (ckpt best_ce)':<30} {ce_auc_test:>10.4f}")
print(f"{'FITB-Accuracy (ckpt best_fitb)':<30} {fitb_acc_test:>10.4f}   (baseline aleatorio: 0.2500)")
print("=" * 55)
 
run_summary_test = {
    "ce_auc_test": ce_auc_test,
    "fitb_acc_test": fitb_acc_test,
    "ce_checkpoint_epoch": None,  # rellenar si quieres guardar epoch/best_metric de cada ckpt
    "fitb_checkpoint_epoch": None,
}
with open(CKPT_DIR + "test_results.json", "w") as f:
    json.dump(run_summary_test, f, indent=2)
print(f"\nResumen guardado en: {CKPT_DIR}test_results.json")

In [ ]:
import json, os

# Reconstruido directamente a partir del log impreso en la celda del bucle de entrenamiento
# (38 épocas -- el entrenamiento se corta ahí, sin la línea final "Mejor Val FITB Acc...",
# así que probablemente se interrumpió a mitad de la época 38/39)
history = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38],
    "train_loss_ce": [0.6627, 0.6189, 0.6014, 0.5817, 0.5652, 0.5506, 0.5397, 0.5308, 0.5194, 0.505, 0.4977, 0.4924, 0.4856, 0.4857, 0.4779, 0.4719, 0.4692, 0.4634, 0.4606, 0.4553, 0.4511, 0.4485, 0.4446, 0.4413, 0.4372, 0.4337, 0.4319, 0.4295, 0.4269, 0.4213, 0.4184, 0.4165, 0.4154, 0.4093, 0.4087, 0.4054, 0.4017, 0.4012],
    "train_loss_fitb": [1.3355, 1.2918, 1.2751, 1.2572, 1.227, 1.2067, 1.1969, 1.1829, 1.1578, 1.1358, 1.1234, 1.1145, 1.1079, 1.1002, 1.0944, 1.0876, 1.0831, 1.0727, 1.0633, 1.0514, 1.0453, 1.0378, 1.0337, 1.0287, 1.0227, 1.0172, 1.0109, 1.0079, 1.0014, 0.9959, 0.9912, 0.9842, 0.9785, 0.9728, 0.9647, 0.9584, 0.9512, 0.9456],
    "train_ce_auc": [0.7123, 0.7376, 0.7508, 0.7809, 0.7971, 0.8106, 0.82, 0.8275, 0.8259, 0.8462, 0.85, 0.8551, 0.8567, 0.8579, 0.8631, 0.8687, 0.8694, 0.8762, 0.8778, 0.8809, 0.8752, 0.8828, 0.8893, 0.8885, 0.8936, 0.8946, 0.8949, 0.8983, 0.9005, 0.8983, 0.9033, 0.9043, 0.9072, 0.9069, 0.9066, 0.9109, 0.9125, 0.912],
    "val_ce_auc": [0.7145, 0.741, 0.7499, 0.782, 0.8013, 0.8153, 0.8224, 0.8296, 0.8269, 0.8465, 0.8495, 0.8519, 0.8532, 0.8553, 0.8569, 0.8626, 0.866, 0.8686, 0.8698, 0.872, 0.8675, 0.8749, 0.88, 0.8791, 0.8828, 0.8841, 0.8829, 0.8837, 0.8878, 0.8858, 0.8904, 0.8911, 0.8921, 0.8923, 0.8907, 0.8942, 0.8938, 0.8928],
    "train_fitb_acc": [0.351, 0.3864, 0.3998, 0.427, 0.44, 0.4596, 0.4568, 0.4758, 0.4856, 0.5026, 0.51, 0.5084, 0.5212, 0.516, 0.5144, 0.5252, 0.5232, 0.533, 0.5382, 0.5456, 0.5492, 0.5572, 0.5564, 0.554, 0.5666, 0.566, 0.5708, 0.5722, 0.5738, 0.5792, 0.5864, 0.586, 0.5884, 0.5918, 0.5904, 0.6022, 0.6032, 0.6064],
    "val_fitb_acc": [0.3584, 0.3782, 0.3956, 0.4202, 0.4412, 0.4596, 0.4648, 0.4804, 0.4958, 0.4996, 0.5154, 0.522, 0.519, 0.5218, 0.5148, 0.5266, 0.53, 0.5354, 0.5464, 0.5472, 0.5508, 0.555, 0.5486, 0.5564, 0.56, 0.5584, 0.5562, 0.5638, 0.5592, 0.5564, 0.5632, 0.5646, 0.5662, 0.5664, 0.561, 0.5644, 0.5654, 0.5612],
}

CKPT_DIR = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/"   # ajusta si tu ruta es otra
os.makedirs(CKPT_DIR, exist_ok=True)
with open(CKPT_DIR + "history.json", "w") as f:
    json.dump(history, f, indent=2)

import numpy as np  # necesario para la celda de las gráficas
print(f"history reconstruido y guardado ({len(history['epoch'])} épocas).")

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Configuración del estilo académico global
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 8.5,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "axes.edgecolor": "0.25",
        "figure.dpi": 300,
    }
)

# Carga de datos
epochs = np.array(history["epoch"])
loss_ce = np.array(history["train_loss_ce"])
loss_fitb = np.array(history["train_loss_fitb"])
ce_auc_train = np.array(history["train_ce_auc"])
ce_auc_val = np.array(history["val_ce_auc"])
fitb_acc_train = np.array(history["train_fitb_acc"])
fitb_acc_val = np.array(history["val_fitb_acc"])

stop_epoch = epochs[-1]
best_ce_epoch = epochs[int(np.argmax(ce_auc_val))]
best_ce_auc = max(ce_auc_val)
best_fitb_epoch = epochs[int(np.argmax(fitb_acc_val))]
best_fitb_acc = max(fitb_acc_val)

# Definición semántica de colores
COLOR_VAL = "#1f4e79"        # Azul para Validación
COLOR_TRAIN = "#d95f02"      # Naranja para Entrenamiento
COLOR_FITB_LOSS = "#4dad78"  # Verde para la segunda pérdida de entrenamiento (Excepción Panel a)


def apply_shared_style(ax):
    """Aplica formato consistente de ejes y marcas de early stopping."""
    ax.axvline(
        stop_epoch,
        color="0.5",
        lw=1.0,
        ls=":",
        label="Parada anticipada" if ax == axes[0] else "",
    )
    # Marcas de época en el eje X
    step = max(1, len(epochs) // 10)
    ax.set_xticks(epochs[::step])
    ax.set_xlim(epochs[0] - 0.5, epochs[-1] + 0.8)
    ax.set_xlabel("Época")


# Crear figura integrada con 3 subfiguras (1 fila x 3 columnas)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharex=False)

# =========================================================
# Panel (a): Pérdidas de entrenamiento (Excepción: Azul vs Verde)
# =========================================================
ax_a = axes[0]
ax_a.plot(
    epochs,
    loss_ce,
    color=COLOR_VAL,
    ls="-",
    marker="o",
    ms=3.5,
    lw=1.4,
    label=r"Pérdida $\mathcal{L}_{\mathrm{CE}}$",
)
ax_a.plot(
    epochs,
    loss_fitb,
    color=COLOR_FITB_LOSS,
    ls="--",
    marker="^",
    ms=3.5,
    lw=1.4,
    label=r"Pérdida $\mathcal{L}_{\mathrm{FITB}}$",
)
ax_a.set_ylabel("Pérdida (Entrenamiento)")
ax_a.set_title(r"(a) Evolución de Pérdidas", loc="left", fontweight="bold")
apply_shared_style(ax_a)
ax_a.legend(loc="upper right", frameon=False)

# =========================================================
# Panel (b): CE-AUC (Naranja = Train, Azul = Val)
# =========================================================
ax_b = axes[1]
ax_b.plot(
    epochs,
    ce_auc_train,
    color=COLOR_TRAIN,
    ls="-",
    marker="o",
    ms=3.5,
    lw=1.4,
    label="AUC Entrenamiento",
)
ax_b.plot(
    epochs,
    ce_auc_val,
    color=COLOR_VAL,
    ls="--",
    marker="s",
    ms=3.5,
    lw=1.4,
    label="AUC Validación",
)
ax_b.scatter(
    [best_ce_epoch],
    [best_ce_auc],
    color="gold",
    s=80,
    marker="*",
    edgecolor="black",
    linewidth=0.8,
    zorder=5,
    label=r"Óptimo Val.",
)

# Anotación adaptativa de valor óptimo
ax_b.annotate(
    f"Máx: {best_ce_auc:.4f}\n(Época {best_ce_epoch})",
    xy=(best_ce_epoch, best_ce_auc),
    xytext=(best_ce_epoch - len(epochs) * 0.25, best_ce_auc - 0.18),
    fontsize=8,
    color="black",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.9),
    arrowprops=dict(
        arrowstyle="->", connectionstyle="arc3,rad=0.2", color="0.3", lw=0.8
    ),
)
ax_b.set_ylim(0.4, 1.0)
ax_b.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax_b.set_ylabel("Compatibilidad (CE-AUC)")
ax_b.set_title(r"(b) Evaluación CE-AUC", loc="left", fontweight="bold")
apply_shared_style(ax_b)
ax_b.legend(loc="lower right", frameon=False)

# =========================================================
# Panel (c): FITB-Accuracy (Naranja = Train, Azul = Val)
# =========================================================
ax_c = axes[2]
ax_c.plot(
    epochs,
    fitb_acc_train,
    color=COLOR_TRAIN,
    ls="-",
    marker="o",
    ms=3.5,
    lw=1.4,
    label="Precisión Entr.",
)
ax_c.plot(
    epochs,
    fitb_acc_val,
    color=COLOR_VAL,
    ls="--",
    marker="s",
    ms=3.5,
    lw=1.4,
    label="Precisión Val.",
)
ax_c.axhline(
    0.25, color="0.5", lw=1.0, ls="-.", label=r"Azar ($0{,}25$)"
)
ax_c.scatter(
    [best_fitb_epoch],
    [best_fitb_acc],
    color="gold",
    s=80,
    marker="*",
    edgecolor="black",
    linewidth=0.8,
    zorder=5,
)

# Anotación adaptativa de valor óptimo
ax_c.annotate(
    f"Máx: {best_fitb_acc:.4f}\n(Época {best_fitb_epoch})",
    xy=(best_fitb_epoch, best_fitb_acc),
    xytext=(best_fitb_epoch - len(epochs) * 0.25, best_fitb_acc - 0.22),
    fontsize=8,
    color="black",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.9),
    arrowprops=dict(
        arrowstyle="->", connectionstyle="arc3,rad=0.2", color="0.3", lw=0.8
    ),
)
ax_c.set_ylim(0.0, 1.0)
ax_c.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax_c.set_ylabel("FITB-Accuracy")
ax_c.set_title(
    r"(c) Evaluación Fill-in-the-Blank", loc="left", fontweight="bold"
)
apply_shared_style(ax_c)
ax_c.legend(loc="lower right", frameon=False)

# Ajuste de diseño final y exportación vectorial
plt.tight_layout()
os.makedirs("./figuras", exist_ok=True)
plt.savefig(
    "./figuras/convergencia_recommendation_engine.pdf", bbox_inches="tight"
)
plt.savefig(
    "./figuras/convergencia_recommendation_engine.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()